# LangChain APIs & Environment Setup

## Introduction

Before building LangChain applications, we need to prepare our Python environment and configure access to an LLM.

In this workshop, we will use:

* **Python**
* **LangChain**
* **ChatOpenAI**
* **OpenRouter**
* **python-dotenv**
* **Chroma** for vector storage
* **FAISS** for vector search

Although we are using the `ChatOpenAI` class, the actual model requests in this workshop will be routed through **OpenRouter** using its OpenAI-compatible API.

Our architecture is:

```text
Python Application
       ↓
    LangChain
       ↓
   ChatOpenAI
       ↓
   OpenRouter API
       ↓
   Selected LLM
```

---

# Step 1: Create a Python Virtual Environment

A virtual environment keeps project dependencies isolated from other Python projects.

Open your terminal in the project directory:

```bash
python -m venv .venv
```

This creates:

```text
.venv/
```

inside your project.

---

# Step 2: Activate the Virtual Environment

## Windows PowerShell

```powershell
.venv\Scripts\Activate.ps1
```

After activation, you should see something similar to:

```text
(.venv) PS C:\your-project>
```

## Windows Command Prompt

```cmd
.venv\Scripts\activate
```

## macOS / Linux

```bash
source .venv/bin/activate
```

---

# Step 3: Upgrade pip

Upgrade `pip` before installing the project dependencies:

```bash
python -m pip install --upgrade pip
```

You can verify the installation:

```bash
pip --version
```

---

# Step 4: Install LangChain Packages

For this workshop, install:

```bash
pip install langchain langchain-core langchain-community langchain-openai python-dotenv openai
```

### What does each package do?

| Package               | Purpose                                         |
| --------------------- | ----------------------------------------------- |
| `langchain`           | Main LangChain framework                        |
| `langchain-core`      | Core abstractions such as prompts and runnables |
| `langchain-community` | Community integrations                          |
| `langchain-openai`    | `ChatOpenAI` integration                        |
| `python-dotenv`       | Loads variables from `.env`                     |
| `openai`              | OpenAI-compatible client dependency             |

---

# Step 5: Install Vector Database Dependencies

For later RAG exercises, install Chroma:

```bash
pip install chromadb
```

You can also install FAISS:

```bash
pip install faiss-cpu
```

These will be useful when we move from simple LLM calls to vector search and RAG.

---

# Step 6: Create the `.env` File

Create a file named:

```text
.env
```

in the root directory of your project.

Our project can look like:

```text
langchain-workshop/
│
├── .venv/
├── .env
├── .gitignore
├── main.py
└── requirements.txt
```

---

# Step 7: Configure the OpenRouter API Key

Because we are using OpenRouter, our `.env` file should contain:

```env
OPENROUTER_API_KEY=sk-or-v1-your_api_key_here
```

You can optionally store the base URL in the `.env` file as well:

```env
OPENROUTER_API_KEY=sk-or-v1-your_api_key_here
OPENROUTER_BASE_URL=https://openrouter.ai/api/v1
```

This keeps the configuration separate from the Python code.

> **Important:** Never share your API key publicly or commit `.env` to GitHub.

---

# Step 8: Add `.env` to `.gitignore`

Create:

```text
.gitignore
```

and add:

```text
.env
.venv/
__pycache__/
```

This prevents sensitive credentials and the virtual environment from being committed to Git.

---

# Step 9: Load Environment Variables

Install `python-dotenv` if you haven't already:

```bash
pip install python-dotenv
```

Then:

```python
from dotenv import load_dotenv
import os

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL")
```

We can check whether the key was loaded:

```python
if OPENROUTER_API_KEY:
    print("API key loaded successfully.")
else:
    print("API key not found.")
```

### Never do this:

```python
print(OPENROUTER_API_KEY)
```

because that exposes your secret API key.

---

# Step 10: Initialize ChatOpenAI with OpenRouter

Now we can initialize `ChatOpenAI`.

```python
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.7,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)
```

The important point here is that:

```python
ChatOpenAI
```

does not necessarily mean that the request must go directly to OpenAI.

We are configuring it to use OpenRouter's OpenAI-compatible API:

```text
ChatOpenAI
     ↓
base_url
     ↓
https://openrouter.ai/api/v1
     ↓
OpenRouter
     ↓
Selected Model
```

---

# Step 11: Use the Base URL from `.env`

Instead of hard-coding the URL, we can use the environment variable.

`.env`:

```env
OPENROUTER_API_KEY=sk-or-v1-your_api_key_here
OPENROUTER_BASE_URL=https://openrouter.ai/api/v1
```

Python:

```python
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os

load_dotenv()

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.7,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)
```

This is the recommended approach for our workshop because configuration is separated from application code.

---

# Step 12: Make Your First LLM Call

Now let's send a simple prompt.

```python
response = llm.invoke(
    "Explain LangChain in one sentence."
)

print(response.content)
```

The response will contain an AI-generated explanation.

For example:

```text
LangChain is a framework for building applications that connect
language models with prompts, tools, data sources and workflows.
```

The exact response can vary because it is generated by the model.

---

# Understanding `.invoke()`

The `.invoke()` method sends input to the LangChain component and returns the result.

```python
response = llm.invoke("Hello!")
```

The flow is:

```text
Python Code
    ↓
.invoke()
    ↓
ChatOpenAI
    ↓
OpenRouter
    ↓
LLM
    ↓
AI Response
```

The response is an `AIMessage`.

To access the generated text:

```python
print(response.content)
```

---

# Step 13: Using Messages

LangChain works with chat messages.

For example:

```python
from langchain_core.messages import HumanMessage

response = llm.invoke([
    HumanMessage(content="Explain LangChain in one sentence.")
])

print(response.content)
```

This represents a human message being sent to the model.

LangChain also supports other message types such as:

```text
HumanMessage
AIMessage
SystemMessage
ToolMessage
```

These become particularly important when building chatbots and agents.

---

# Step 14: Complete OpenRouter + ChatOpenAI Example

Create:

```text
main.py
```

Add:

```python
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os


# Load environment variables
load_dotenv()


# Initialize ChatOpenAI through OpenRouter
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.7,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)


# Send a prompt
response = llm.invoke(
    "Explain LangChain in one sentence."
)


# Display the response
print("AI Response:")
print(response.content)
```

Run:

```bash
python main.py
```

---

# Step 15: Understanding the Configuration

Let's understand this important section:

```python
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.7,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)
```

## `model`

```python
model="openai/gpt-4o-mini"
```

Specifies which model OpenRouter should use.

The model name must be one supported by OpenRouter.

You can change the model without changing the rest of the LangChain code.

---

## `temperature`

```python
temperature=0.7
```

Controls the randomness of the model's responses.

A simplified interpretation:

```text
0.0 → More deterministic
0.3 → Less random
0.7 → More creative
1.0 → Higher randomness
```

For factual tasks, lower values can be useful.

For creative tasks, higher values can be useful.

---

## `api_key`

```python
api_key=os.getenv("OPENROUTER_API_KEY")
```

Reads the API key from the `.env` file.

This is better than putting the key directly in your Python source code.

---

## `base_url`

```python
base_url=os.getenv("OPENROUTER_BASE_URL")
```

Tells `ChatOpenAI` to send requests to OpenRouter instead of the default OpenAI endpoint.

Our value is:

```text
https://openrouter.ai/api/v1
```

---

# Step 16: Testing the API Connection

Before continuing with LangChain, test whether your configuration works.

Create:

```python
response = llm.invoke("Say hello in one sentence.")

print(response.content)
```

If everything is configured correctly, you should receive an AI response.

If you get an authentication error, check:

1. `.env` exists in the correct directory.
2. `OPENROUTER_API_KEY` is spelled correctly.
3. The API key is valid.
4. `load_dotenv()` is being called.
5. The virtual environment is activated.
6. The required packages are installed.

---

# Step 17: Common Environment Errors

## Error: `ModuleNotFoundError`

Example:

```text
ModuleNotFoundError: No module named 'langchain_openai'
```

Install:

```bash
pip install langchain-openai
```

---

## Error: API Key Not Found

Check:

```python
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("OPENROUTER_API_KEY") is not None)
```

Expected:

```text
True
```

Do not print the actual key.

---

## Error: `.env` Not Loading

Make sure your project looks like:

```text
project/
│
├── .env
└── main.py
```

and call:

```python
load_dotenv()
```

before accessing the environment variable.

---

# Step 18: Requirements File

Once the environment is working, we can save our dependencies.

Run:

```bash
pip freeze > requirements.txt
```

This creates:

```text
requirements.txt
```

Another developer can then install the dependencies using:

```bash
pip install -r requirements.txt
```

For a workshop project, it is often useful to maintain a requirements file so everyone can reproduce the environment.

---

# Step 19: Vector Store Setup

Once the basic LLM call works, we can install vector-store dependencies:

```bash
pip install chromadb faiss-cpu
```

The purpose of a vector store is to store embeddings and perform similarity search.

The eventual RAG architecture will look like:

```text
Documents
    ↓
Document Loader
    ↓
Text Splitter
    ↓
Embeddings
    ↓
Vector Store
    ↓
Retriever
    ↓
Relevant Documents
    ↓
Prompt
    ↓
ChatOpenAI
    ↓
Response
```

---

# Step 20: Simple Chroma Example

For a vector-store demonstration, we can use OpenRouter-compatible embeddings only if the selected embedding provider/model is supported by the configured setup.

For the introductory workshop, focus first on understanding the architecture:

```text
Text
 ↓
Embedding Model
 ↓
Vector
 ↓
Vector Store
 ↓
Similarity Search
```

The LLM and embedding model are separate concepts.

An LLM generates or interprets language, while an embedding model converts text into numerical vectors for semantic search.

---

# Final Project Structure

At this stage, your workshop project can look like:

```text
langchain-workshop/
│
├── .venv/
│
├── .env
├── .gitignore
├── main.py
└── requirements.txt
```

### `.env`

```env
OPENROUTER_API_KEY=sk-or-v1-your_api_key_here
OPENROUTER_BASE_URL=https://openrouter.ai/api/v1
```

### `.gitignore`

```text
.env
.venv/
__pycache__/
```

---

# Complete Environment Flow

The complete setup is:

```text
                  Python
                    ↓
              Virtual Environment
                    ↓
               Install Packages
                    ↓
                  .env
                    ↓
           OPENROUTER_API_KEY
                    ↓
               ChatOpenAI
                    ↓
              OpenRouter API
                    ↓
             Selected LLM
                    ↓
                Response
```

---

# Key Takeaways

By the end of this section, you should understand:

1. How to create a Python virtual environment.
2. How to activate the environment.
3. How to install LangChain packages.
4. How to use `python-dotenv`.
5. Why API keys should be stored in `.env`.
6. How to configure OpenRouter.
7. How `ChatOpenAI` can communicate through OpenRouter.
8. What `base_url` does.
9. How `.invoke()` works.
10. How to access the response using `.content`.
11. How LangChain can later connect to vector stores and RAG systems.

The most important configuration for this workshop is:

```python
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.7,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)
```

And the corresponding `.env`:

```env
OPENROUTER_API_KEY=sk-or-v1-your_api_key_here
OPENROUTER_BASE_URL=https://openrouter.ai/api/v1
```

From here, we can move to the next LangChain concepts:

```text
ChatOpenAI
    ↓
PromptTemplate
    ↓
OutputParser
    ↓
LCEL
    ↓
Chains
    ↓
RAG
    ↓
Agents
    ↓
LangGraph
```
